In [ ]:
# basic imports
import os
import numpy as np
from tess_backml import BackgroundCube
from tess_backml.utils import plot_img, animate_cube
import matplotlib.pyplot as plt
from IPython.display import HTML
from astropy.time import Time
import functools

# increase animation frame limits
import matplotlib
matplotlib.rcParams["animation.embed_limit"] = 2**128

In [ ]:
from astropy.visualization import simple_norm
from matplotlib import animation, axes, colors
from typing import Callable, Optional, Tuple, Union

def plot_img(
    tdx: int,
    extent: Optional[Tuple] = None,
    cbar: bool = True,
    ax: Optional[plt.Axes] = None,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    cnorm: Optional[colors.Normalize] = None,
    plot_asteroids: bool = False,
    vmag_lim: float = 21.0,
) -> axes.Axes:
    """
    Plots an image with optional scatter points and colorbar.

    Parameters
    ----------
    img : np.ndarray
        The 2D image data to be plotted.
    scol_2d : Optional[np.ndarray], optional
        The column coordinates of scatter points, by default None.
    srow_2d : Optional[np.ndarray], optional
        The row coordinates of scatter points, by default None.
    plot_type : str, optional
        The type of plot to create ('img' or 'scatter'), by default "img".
    extent : Optional[Tuple], optional
        The extent of the image (left, right, bottom, top), by default None.
    cbar : bool, optional
        Whether to display a colorbar, by default True.
    ax : Optional[plt.Axes], optional
        The matplotlib Axes object to plot on, by default None. If None, a new figure and axes are created.
    title : str, optional
        The title of the plot, by default "".
    vmin : Optional[float], optional
        The minimum value for the colormap, by default None.
    vmax : Optional[float], optional
        The maximum value for the colormap, by default None.
    cnorm : Optional[colors.Normalize], optional
        Custom color normalization, by default None.
    bar_label : str, optional
        The label for the colorbar, by default "Flux [e-/s]"."

    Returns
    -------
    matplotlib.axes
    """
    # Initialise ax
    if ax is None:
        _, ax = plt.subplots()

    ffi_corr = get_corrected_ffi(tdx=tdx)
    # ffi_corr -= np.nanmedian(ffi_corr)
    # print(tdx)

    iso = Time(cube.tcube.time[tdx] + cube.btjd0, format="jd", scale="tdb").iso[:-4]
    title = f"CAD {cube.tcube.cadence_number[tdx]} | Time {iso}"

    # Define vmin and vmax
    if cnorm is None and vmin is None and vmax is None:
        vmin, vmax = np.nanpercentile(ffi_corr.ravel(), [40, 70])

    # Plot image, colorbar and marker
    im = ax.imshow(
        ffi_corr,
        origin="lower",
        cmap="Greys",
        vmin=vmin,
        vmax=vmax,
        norm=cnorm,
        extent=extent,
        rasterized=True,
    )
    if cbar:
        plt.colorbar(im, location="right", shrink=0.8).set_label(label="Flux [$e^-/s$ ]", size=14)

    if plot_asteroids:
        mask = average_mag.values.ravel() <= vmag_lim
        ax.scatter(
            pos_col.values[mask, tdx] - cube.cmin, 
            pos_row.values[mask, tdx] - cube.rmin, 
            s=50, 
            facecolors='none', 
            edgecolors='tab:red'
        )

    ax.set_aspect("equal", "box")
    ax.set_title(title, fontsize=16)

    # ax.set_xlabel("Pixel Column")
    # ax.set_ylabel("Pixel Row")
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False)
    ax.set_xlim(0, ffi_corr.shape[1])
    ax.set_ylim(0, ffi_corr.shape[0])

    return ax

def animate_cube(
    cadenceno: np.ndarray,
    extent: Optional[Tuple] = None,
    interval: int = 200,
    repeat_delay: int = 1000,
    step: int = 1,
    suptitle: str = "",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    plot_asteroids: bool = False,
    vmag_lim: float = 21.0,
    cbar: bool = True,
) -> animation.FuncAnimation:
    """
    Animates a 3D data cube (time series of 2D images).

    Parameters
    ----------
    cube : np.ndarray
        The 3D data cube to be animated (time, row, column).
    scol_2d : Optional[np.ndarray], optional
        The column coordinates of scatter points, by default None.
    srow_2d : Optional[np.ndarray], optional
        The row coordinates of scatter points, by default None.
    cadenceno : Optional[np.ndarray], optional
        Cadence numbers corresponding to the time axis, by default None.
    time : Optional[np.ndarray], optional
        Time values corresponding to the time axis, by default None.
    plot_type : str, optional
        The type of plot to create ('img' or 'scatter'), by default "img".
    extent : Optional[Tuple], optional
        The extent of the images (left, right, bottom, top), by default None.
    interval : int, optional
        Delay between frames in milliseconds, by default 200.
    repeat_delay : int, optional
        Delay in milliseconds before repeating the animation, by default 1000.
    step : int, optional
        Step size for frame selection, by default 1.
    suptitle : str, optional
        Overall title for the animation figure, by default "".
    bar_label : str, optional
        The label for the colorbar, by default "Flux [e-/s]".

    Returns
    -------
    animation.FuncAnimation
        The matplotlib FuncAnimation object.
    """
    # Initialise figure and set title
    fig, ax = plt.subplots(figsize=(12, 12))
    fig.suptitle(suptitle, y=1.001, fontsize=20)
    plt.tight_layout()

    # if vmin is None and vmax is None:
    #     vmin=-10
    #     vmax=10

    # Plot first image in cube.
    nt = 0

    ax = plot_img(
        nt,
        extent=extent,
        cbar=cbar,
        ax=ax,
        vmin=vmin,
        vmax=vmax,
        plot_asteroids=plot_asteroids,
        vmag_lim=vmag_lim,
    )

    # Define function for animation
    def animate(nt):
        ax.clear()
        _ = plot_img(
            nt,
            extent=extent,
            cbar=cbar,
            ax=ax,
            vmin=vmin,
            vmax=vmax,
            plot_asteroids=plot_asteroids,
            vmag_lim=vmag_lim,
        )

        return ()

    # Prevent second figure from showing up in interactive mode
    plt.close(ax.figure)  # type: ignore

    # Create the animation
    ani = animation.FuncAnimation(
        fig,
        animate,
        frames=cadenceno[::step],
        interval=interval,
        blit=True,
        repeat_delay=repeat_delay,
        repeat=True,
    )

    return ani

In [ ]:
# define sector/camera/ccd
sector = 3
camera = 1
ccd = 4

In [ ]:
cube = BackgroundCube(
    sector=sector, camera=camera, ccd=ccd, img_bin=1, downsize="binning"
)

In [ ]:
cube._get_dark_frame_idx()
cube._get_star_mask(sigma=5.0, dilat_iter=2)

In [ ]:
tdx = np.arange(0, cube.nt, 100)

ffi_cube = np.array([cube.tcube.get_ffi(k)[1].data[
    cube.rmin : cube.rmax, cube.cmin : cube.cmax
    ] for k in tdx]
                   )
ffi_cube.shape

In [ ]:
from tess_backml.corrector import ScatterLightCorrector
import os

In [ ]:
# Corrector object
slcorr = ScatterLightCorrector(
    sector=sector, camera=camera, ccd=ccd, 
    fname=f"/Users/jimartin/Work/TESS/tess-backml/data/cubes/sector{sector:03}/ffi_slcube_sector{sector:03}_{camera}-{ccd}.fits"
)

In [ ]:
cadno = np.arange(cube.tcube.cadence_number[cube.tcube.quality == 0].shape[0])
times = cube.tcube.time[cadno] + cube.btjd0
cadno.shape, times.shape

In [ ]:
step = 16
srow = np.arange(
    cube.rmin + step / 2,
    cube.rmax - step / 2,
    step,
    dtype=int,
)
scol = np.arange(
    cube.cmin + step / 2,
    cube.cmax - step / 2,
    step,
    dtype=int,
)
srow_2d, scol_2d = np.meshgrid(srow, scol, indexing="ij")
sparse_rc = [(r, c) for r, c in zip(srow_2d.ravel(), scol_2d.ravel())][::2]
pix_ts = cube.tcube.get_pixel_timeseries(sparse_rc, return_errors=False)

# reshape as [ntimes, npix]
pix_ts = pix_ts.reshape((cube.tcube.nframes, -1))
# take median
cube.bkg_lc = np.nanmedian(pix_ts, axis=-1)

In [ ]:
from astropy.stats import sigma_clip

In [ ]:
bkg_med = np.median(cube.bkg_lc)
bkg_std = np.std(cube.bkg_lc)
bkg_per = np.percentile(cube.bkg_lc, 85)
# sl_frames = sigma_clip(cube.bkg_lc, sigma_upper=3, sigma_lower=10, masked=True).mask

sl_frames = cube.bkg_lc > bkg_per
good_mask = (cube.tcube.quality == 0) & ~sl_frames
good_cads = np.arange(good_mask.shape[0])[good_mask]

plt.plot(cube.tcube.time, cube.bkg_lc, ".", ms=1)
plt.plot(cube.tcube.time[cube.tcube.quality == 0], cube.bkg_lc[cube.tcube.quality == 0], ".", ms=1)
plt.plot(cube.tcube.time[good_mask], cube.bkg_lc[good_mask], ".", ms=1)
plt.show()

In [ ]:
datagaps_idx = np.where(np.diff(cube.tcube.time[good_mask]) > 0.1)[0] + 1
datagaps_cdx = np.array([seg[0] for seg in np.array_split(good_cads, datagaps_idx)[1:]])
datagaps_cdx = np.concatenate([[cube.tcube.cadence_number.min()], datagaps_cdx, [cube.tcube.cadence_number.shape[0]-1]])
print(datagaps_idx, datagaps_cdx)

dark_frames = [seg[np.argsort(cube.bkg_lc[seg])[:10]] for seg in np.array_split(good_cads, datagaps_idx)]
dark_frames = []
datagaps_cdx2 = []
for seg in np.array_split(good_cads, datagaps_idx):
    start = seg[::75]
    chunks = [np.arange(s, s + 5) for s in start]
    chunks = [x[np.isin(x, good_cads)] for x in chunks]
    dark_frames.extend(chunks)
    datagaps_cdx2.extend([x[0] for x in chunks])
# dark_frames = np.concatenate(dark_frames)
datagaps_cdx2 = np.unique(np.concatenate([[cube.tcube.cadence_number.min()], datagaps_cdx2, [cube.tcube.cadence_number.shape[0]-1]]))
dark_frames

In [ ]:
plt.vlines(cube.tcube.cadence_number[datagaps_cdx], 
           ymin=cube.tcube.time[good_mask].min(), 
           ymax=cube.tcube.time[good_mask].max(), 
           color="k", lw=1)
plt.vlines(cube.tcube.cadence_number[datagaps_cdx2], 
           ymin=cube.tcube.time[good_mask].min(), 
           ymax=cube.tcube.time[good_mask].max(), 
           color="k", ls="--", lw=1)
plt.scatter(cube.tcube.cadence_number[good_mask], cube.tcube.time[good_mask], s=0.5)
for k in range(len(dark_frames)):
    plt.scatter(cube.tcube.cadence_number[dark_frames[k]], cube.tcube.time[dark_frames[k]], s=0.5)

In [ ]:
row_ffi = np.arange(cube.rmin, cube.rmax)
col_ffi = np.arange(cube.cmin, cube.cmax)

@functools.lru_cache(maxsize=128)
def get_static_scene(self, frames):
    static = np.median(
        [
            self.tcube.get_ffi(f)[1].data[
                self.rmin : self.rmax, self.cmin : self.cmax
            ] - slcorr.evaluate_scatterlight_model(
                row_eval=row_ffi, 
                col_eval=col_ffi, 
                times=cube.tcube.time[[f]] + cube.btjd0
            )[0][0]
            for f in frames
        ],
        axis=0,
    )

    return static

In [ ]:
if os.path.isfile(f"./data/sector{sector:03}_{camera}-{ccd}_stars_static.npz"):
    aux = np.load(f"./data/sector{sector:03}_{camera}-{ccd}_stars_static.npz")
    static = aux["arr_0"]
    datagaps_cdx2 = aux["arr_1"]
else:
    static = np.array([
        get_static_scene(cube, tuple(x)) for x in dark_frames
    ])
    np.savez(f"./data/sector{sector:03}_{camera}-{ccd}_stars_static.npz", static, datagaps_cdx2)

In [ ]:
for sta in static[:5]:
    bar = plt.imshow(static[0], vmin=100, vmax=500, origin="lower")
    plt.colorbar(bar)
    plt.show()

In [ ]:
@functools.lru_cache(maxsize=128)
def get_corrected_ffi(tdx=tdx):
    flx = cube.tcube.get_ffi(tdx)[1].data[cube.rmin : cube.rmax, cube.cmin : cube.cmax]
    
    sl = slcorr.evaluate_scatterlight_model(
        row_eval=row_ffi, 
        col_eval=col_ffi, 
        times=cube.tcube.time[[tdx]] + cube.btjd0
    )[0][0]
    
    for k, (a, b) in enumerate(zip(datagaps_cdx2[:-1], datagaps_cdx2[1:])):
        if tdx == datagaps_cdx2[-1]:
            b += 1
        if tdx >= a and tdx < b:
            static_idx = k
    if static_idx == len(static):
        static_idx -= 1

    return flx - static[static_idx] - sl

In [ ]:
test = get_corrected_ffi(654)

bar = plt.imshow(test, vmin=-10, vmax=10, origin="lower")
plt.colorbar(bar)
plt.show()

In [ ]:
plot_img(654, vmin=-2, vmax=10);

In [ ]:
from glob import glob
import pandas as pd
from tqdm import tqdm

In [ ]:
fnames = sorted(glob(f"/Users/jimartin/Work/TESS/tess-asteroid-ml/data/jpl/tracks/sector{sector:04}/tess-ffi_s{sector:04}-*_hires.feather"))
len(fnames)

pos_row = pd.DataFrame(columns=cube.tcube.time + cube.btjd0)
pos_col = pd.DataFrame(columns=cube.tcube.time + cube.btjd0)
average_mag = []

for k, f in tqdm(enumerate(fnames), total=len(fnames)):
    track = pd.read_feather(f)
    if len(track.query(f"camera == {camera} and ccd == {ccd} and vmag <= 21")) > 0:
        tr_row = np.interp(cube.tcube.time + cube.btjd0, track.time, track.row, left=np.nan, right=np.nan)
        tr_col = np.interp(cube.tcube.time + cube.btjd0, track.time, track.column, left=np.nan, right=np.nan)
        pos_row.loc[k] = tr_row
        pos_col.loc[k] = tr_col
        average_mag.append(track.vmag.mean())
    # break
average_mag = np.array(average_mag)
average_mag = pd.DataFrame(average_mag, columns=["vmag"])

# pos_row.to_feather(f"sector{sector:03}_{camera}-{ccd}_all_pos_row.feather")
# pos_col.to_feather(f"sector{sector:03}_{camera}-{ccd}_all_pos_col.feather")
# average_mag.to_feather(f"sector{sector:03}_{camera}-{ccd}_all_vmag.feather")

In [ ]:
sector, camera, ccd

In [ ]:
pos_row

In [ ]:
mask = average_mag.values.ravel() < 18
plt.scatter(pos_col.values[mask, 1], pos_row.values[mask, 1], s=80, facecolors='none', edgecolors="tab:red");

In [ ]:
sector, camera, ccd

In [ ]:
total = good_cads.shape[0]
step = 10
vmag_lim = 20
good_cads[:total][::step].shape

In [ ]:
(average_mag.values.ravel() < vmag_lim).sum()

In [ ]:
good_cads[:total][::step]

In [ ]:
ani = animate_cube(
        good_cads[:total],
        vmin=-1,
        vmax= 8,
        step=step,
        suptitle=f"TESS Sector {sector} Camera {camera} CCD {ccd}\n Asteroids V < {vmag_lim}",
        plot_asteroids=True,
        vmag_lim=vmag_lim,
        cbar=False,
        )
HTML(ani.to_jshtml())

In [ ]:
file_name = f"./figures/sector{sector:03}_{camera}-{ccd}_asteroid_v{vmag_lim}_ani_t{total:04}_s{step:02}_v2.gif"
ani.save(file_name, writer="pillow")
file_name

In [ ]:
total